# 0.0 - Data Observation

`SupportData.xlsx` is the main claim/support input file and also observes the supporting finance files `GLAccountData.xlsx` and `GLEntryData.xlsx`. The customer claim view and supplier support view are derived from the support dataset, while the GL account and entry data provide account-balance and credit-entry evidence for the project.

Load in the data


In [1]:
import pandas as pd
from pathlib import Path

support_file = Path("SupportData.xlsx")
gl_account_file = Path("GLAccountData.xlsx")
gl_entry_file = Path("GLEntryData.xlsx")

for required_file in [support_file, gl_account_file, gl_entry_file]:
    if not required_file.exists():
        raise FileNotFoundError(f"Missing required source file: {required_file}")

support = pd.read_excel(support_file)

print("Source files used: SupportData.xlsx, GLAccountData.xlsx, GLEntryData.xlsx")
print("SupportData.xlsx shape:", support.shape)
display(support.head())


Source files used: SupportData.xlsx, GLAccountData.xlsx, GLEntryData.xlsx
SupportData.xlsx shape: (73320, 41)


,Date,Month,Year,SalesDeliveryNote.Branch,Customer,Customer.Name,Product,Product.ManufacturerProductCode,Contract.Number,SourceTransactionType,...,SalesDeliveryNote.NetAmountLessDiscountBase,Product.CSQL_InvoiceCostForbranch,Product.CSQL_ListPriceForBranch,Product.CALC_FixedCostForBranch,Contract.Description,Contract.Expression,Contract.ValidFrom,Contract.ValidTo,Contract.Supplier.Code,Contract.Supplier.Name
0,26/02/2024,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS1228,38300,86.0,SL/Del,...,392.18,6.235348,14.21,4.614158,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
1,26/02/2024,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS12S28,38322,86.0,SL/Del,...,392.18,8.130964,18.53,6.016913,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
2,26/02/2024,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS62822,38204,86.0,SL/Del,...,392.18,5.432344,12.38,4.019935,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
3,26/02/2024,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252815,38492,86.0,SL/Del,...,392.18,11.351756,25.87,8.400299,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
4,26/02/2024,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252215,38490,86.0,SL/Del,...,392.18,4.058900,9.25,3.003586,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1


The support file contains both the customer claim-line fields and the supplier contract/support fields. 

Then clean the column names the column names and creates two derived views from `support`: `cust` for claim-line observation and `supp` for supplier-support observation.


In [2]:
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(".", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.replace(r"_+", "_", regex=True)
    )
    return df

support_clean = clean_columns(support)

if "Date" in support_clean.columns:
    support_clean["Date"] = pd.to_datetime(support_clean["Date"], errors="coerce", dayfirst=True)

cust = support_clean.copy()

supplier_key_cols = [
    "Date",
    "Contract_Number",
    "Contract_D_ContractNumber",
    "Product_ManufacturerProductCode",
    "Product_Category",
    "Contract_Description",
    "Contract_Expression",
    "Contract_ValidFrom",
    "Contract_ValidTo",
    "Contract_Supplier_Code",
    "Contract_Supplier_Name",
    "UnitClaimAmount",
    "TotalClaimAmount"
]
supp = support_clean[[col for col in supplier_key_cols if col in support_clean.columns]].drop_duplicates()

print("Clean support columns:")
print(support_clean.columns.to_list())
print("Customer claim view shape:", cust.shape)
print("Supplier support view shape:", supp.shape)


Clean support columns:
['Date', 'Month', 'Year', 'SalesDeliveryNote_Branch', 'Customer', 'Customer_Name', 'Product', 'Product_ManufacturerProductCode', 'Contract_Number', 'SourceTransactionType', 'Contract_D_ContractNumber', 'SalesInvoice_Number', 'SalesCreditNote_Number', 'SalesDeliveryNote_Number', 'SalesReturnNote_Number', 'SalesOrder_Number', 'Status', 'UnitClaimAmount', 'TotalClaimAmount', 'SalesDeliveryNoteLine_Quantity', 'SalesInvoiceLine_Quantity', 'SalesInvoice_Branch', 'SalesDeliveryNoteLine_DiscountPercentage', 'Product_Category', 'SalesCreditNoteLine_D_InvoiceCost', 'SalesCreditNoteLine_D_FixedCost', 'SalesDeliveryNoteLine_D_InvoiceCost', 'SalesDeliveryNoteLine_D_FixedCost', 'SalesInvoiceLine_D_InvoiceCost', 'SalesInvoiceLine_D_FixedCost', 'SalesDeliveryNote_Branch_1', 'SalesDeliveryNote_NetAmountLessDiscountBase', 'Product_CSQL_InvoiceCostForbranch', 'Product_CSQL_ListPriceForBranch', 'Product_CALC_FixedCostForBranch', 'Contract_Description', 'Contract_Expression', 'Contra

## GL Account And Entry Observation

`GLAccountData.xlsx` and `GLEntryData.xlsx` are included as supporting finance evidence. They give the GL account balance view and the individual entry-line credit movements


In [3]:
account_raw = clean_columns(pd.read_excel(gl_account_file, dtype=str))
entry_raw = clean_columns(pd.read_excel(gl_entry_file, dtype=str))

print("GL account shape - ", account_raw.shape)
print("GL entry shape - ", entry_raw.shape)

print("GL account columns - ")
display(pd.Series(account_raw.columns))

print("GL entry columns - ")
display(pd.Series(entry_raw.columns))

display(account_raw.head())
display(entry_raw.head())

for col in ["CurrentBalance"]:
    if col in account_raw.columns:
        account_raw[col] = pd.to_numeric(account_raw[col], errors="coerce")

for col in ["Total_Balance", "Credit_Amount", "Running_Balance"]:
    if col in entry_raw.columns:
        entry_raw[col] = pd.to_numeric(entry_raw[col], errors="coerce")

if "Entry_Line_Date" in entry_raw.columns:
    entry_raw["Entry_Line_Date"] = pd.to_datetime(entry_raw["Entry_Line_Date"], errors="coerce", dayfirst=True)

print("GL account balance summary:")
display(account_raw[[col for col in ["Code", "Description", "CurrentBalance"] if col in account_raw.columns]].head(10))

print("GL entry credit amount summary:")
display(entry_raw[[col for col in ["GL_Account_Code", "Description", "Entry_Line_Date", "Credit_Amount", "Running_Balance"] if col in entry_raw.columns]].describe(include="all"))


GL account shape -  (21, 4)
GL entry shape -  (170, 6)
GL account columns - 


0                   Code
1            Description
2    D_MerlinAccountCode
3         CurrentBalance
dtype: object

GL entry columns - 


0    GL_Account_Code
1        Description
2      Total_Balance
3    Entry_Line_Date
4      Credit_Amount
5    Running_Balance
dtype: object

,Code,Description,D_MerlinAccountCode,CurrentBalance
0,01-85001,Supplier13 E071 Claimbacks,NaN,-242784.06
1,01-85002,Supplier2 E005 Claimbacks,NaN,-6240
2,01-85003,Supplier2GB E005GB Claimbacks,NaN,0
3,01-85007,Supplier9 E022 Claimbacks,NaN,-5774.89
4,01-85017,Supplier3 E055 Claimbacks,NaN,-5550


,GL_Account_Code,Description,Total_Balance,Entry_Line_Date,Credit_Amount,Running_Balance
0,01-85001,Supplier13 E071 Claimbacks,242784.06,2026-05-01 00:00:00,13348.45,13348.45
1,01-85001,Supplier13 E071 Claimbacks,242784.06,2026-05-01 00:00:00,765,14113.45
2,01-85001,Supplier13 E071 Claimbacks,242784.06,2026-05-01 00:00:00,5397.33,19510.78
3,01-85001,Supplier13 E071 Claimbacks,242784.06,2026-05-01 00:00:00,2035,21545.78
4,01-85001,Supplier13 E071 Claimbacks,242784.06,2026-05-01 00:00:00,1038.92,22584.7


GL account balance summary:


,Code,Description,CurrentBalance
0,01-85001,Supplier13 E071 Claimbacks,-242784.06
1,01-85002,Supplier2 E005 Claimbacks,-6240.00
2,01-85003,Supplier2GB E005GB Claimbacks,0.00
3,01-85007,Supplier9 E022 Claimbacks,-5774.89
4,01-85017,Supplier3 E055 Claimbacks,-5550.00
5,01-85018,Supplier5 E059 Claimbacks,0.00
6,01-85021,Supplier19 E070 Claimbacks,-9124.28
7,01-85036,Supplier8 E175 Claimbacks,-4782.68
8,01-85046,Supplier10 E234 Claimbacks,-925.00
9,01-85056,Supplier17 E269A Claimbacks,-576.00


GL entry credit amount summary:


,GL_Account_Code,Description,Entry_Line_Date,Credit_Amount,Running_Balance
count,170,170,100,170.000000,170.000000
unique,15,15,NaN,NaN,NaN
top,01-85001,Supplier13 E071 Claimbacks,NaN,NaN,NaN
freq,105,105,NaN,NaN,NaN
mean,NaN,NaN,2026-03-01 00:57:36,2198.157176,78717.178471
min,NaN,NaN,2026-01-05 00:00:00,15.000000,17.000000
25%,NaN,NaN,2026-01-05 00:00:00,303.750000,6241.277500
50%,NaN,NaN,2026-03-07 00:00:00,885.000000,47028.325000
75%,NaN,NaN,2026-03-07 00:00:00,2582.060000,149548.430000
max,NaN,NaN,2026-10-07 00:00:00,23765.130000,242784.060000


In [4]:
display(cust.info())
display(supp.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73320 entries, 0 to 73319
Data columns (total 41 columns):
 #   Column                                       Non-Null Count  Dtype         
---  ------                                       --------------  -----         
 0   Date                                         73320 non-null  datetime64[ns]
 1   Month                                        0 non-null      float64       
 2   Year                                         0 non-null      float64       
 3   SalesDeliveryNote_Branch                     54856 non-null  float64       
 4   Customer                                     73320 non-null  object        
 5   Customer_Name                                73320 non-null  object        
 6   Product                                      73320 non-null  object        
 7   Product_ManufacturerProductCode              73317 non-null  object        
 8   Contract_Number                              73319 non-null  float64       


None

<class 'pandas.core.frame.DataFrame'>
Index: 53095 entries, 0 to 73319
Data columns (total 13 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Date                             53095 non-null  datetime64[ns]
 1   Contract_Number                  53094 non-null  float64       
 2   Contract_D_ContractNumber        50068 non-null  object        
 3   Product_ManufacturerProductCode  53092 non-null  object        
 4   Product_Category                 53095 non-null  object        
 5   Contract_Description             53094 non-null  object        
 6   Contract_Expression              46680 non-null  object        
 7   Contract_ValidFrom               53095 non-null  object        
 8   Contract_ValidTo                 53095 non-null  object        
 9   Contract_Supplier_Code           53094 non-null  object        
 10  Contract_Supplier_Name           53094 non-null  object        

None

Mixture of data types across both datasets which will need adjusting.

Most features in the support data have values, with missingness concentrated in transaction references that only apply to some claim lines.


In [5]:
display(cust.describe(include="all"))
display(supp.describe(include="all"))

,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,...,SalesDeliveryNote_NetAmountLessDiscountBase,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name
count,73320,0.0,0.0,54856.000000,73320,73320,73320,73317,73319.000000,73320,...,73320.000000,73320.000000,73320.000000,73320.000000,73319,63340,73320,73320,73319,73319
unique,NaN,NaN,NaN,NaN,863,855,1704,1702,NaN,3,...,NaN,NaN,NaN,NaN,667,120,1,1,25,24
top,NaN,NaN,NaN,NaN,NUG735,Michael Nugent Ltd,YXS1215,38280,NaN,SL/Del,...,NaN,NaN,NaN,NaN,M NUGENT - YX,[D_InvoiceCost]*0.2500,01/01/0001,01/01/0001,U106,Supplier1
freq,NaN,NaN,NaN,NaN,4247,4247,5402,5402,NaN,54836,...,NaN,NaN,NaN,NaN,3894,6136,73320,73320,48112,48112
mean,2025-05-27 23:07:59.607201280,NaN,NaN,7.202384,NaN,NaN,NaN,NaN,404.802671,NaN,...,674.282342,94.818873,126.432358,86.314716,NaN,NaN,NaN,NaN,NaN,NaN
min,2024-02-26 00:00:00,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,2.000000,NaN,...,-4351.460000,0.075200,0.160000,0.072192,NaN,NaN,NaN,NaN,NaN,NaN
25%,2024-11-06 00:00:00,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,85.000000,NaN,...,0.000000,2.501160,5.700000,1.850858,NaN,NaN,NaN,NaN,NaN,NaN
50%,2025-06-11 00:00:00,NaN,NaN,4.000000,NaN,NaN,NaN,NaN,193.000000,NaN,...,171.530000,5.276000,12.380000,4.019935,NaN,NaN,NaN,NaN,NaN,NaN
75%,2025-12-17 00:00:00,NaN,NaN,11.000000,NaN,NaN,NaN,NaN,630.000000,NaN,...,893.045000,22.361248,38.600000,16.995426,NaN,NaN,NaN,NaN,NaN,NaN
max,2026-06-18 00:00:00,NaN,NaN,22.000000,NaN,NaN,NaN,NaN,1903.000000,NaN,...,32982.810000,4324.800000,5440.000000,4260.201600,NaN,NaN,NaN,NaN,NaN,NaN


,Date,Contract_Number,Contract_D_ContractNumber,Product_ManufacturerProductCode,Product_Category,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,UnitClaimAmount,TotalClaimAmount
count,53095,53094.000000,50068,53092,53095,53094,46680,53095,53095,53094,53094,53095.000000,53095.000000
unique,NaN,NaN,366,1702,86,667,120,1,1,25,24,NaN,NaN
top,NaN,NaN,61247,38280,PRSCU,M NUGENT - YX,[D_InvoiceCost]*0.2500,01/01/0001,01/01/0001,U106,Supplier1,NaN,NaN
freq,NaN,NaN,2824,3610,32893,2502,3880,53095,53095,33606,33606,NaN,NaN
mean,2025-05-24 05:39:52.971089408,430.085565,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.099857,37.649891
min,2024-02-26 00:00:00,2.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,-1741.740000
25%,2024-10-31 00:00:00,99.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.369380,1.478100
50%,2025-06-05 00:00:00,193.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.814770,3.836700
75%,2025-12-15 00:00:00,665.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.370304,12.866000
max,2026-06-18 00:00:00,1903.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,526.000000,18500.000000


The only values where the percentiles and statistics from the *info* function include the invoice and fixed costs of the products. The rest are characteristic and will not provide any significant values

In [6]:
display(cust.isna().mean().sort_values(ascending=False))
display(supp.isna().mean().sort_values(ascending=False))

Month                                          1.000000
Year                                           1.000000
SalesReturnNote_Number                         0.999905
SalesCreditNote_Number                         0.976255
SalesInvoice_Number                            0.261443
SalesInvoice_Branch                            0.261443
SalesOrder_Number                              0.251841
SalesDeliveryNote_Number                       0.251828
SalesDeliveryNote_Branch                       0.251828
SalesDeliveryNote_Branch_1                     0.251828
Contract_Expression                            0.136116
Contract_D_ContractNumber                      0.046945
Product_ManufacturerProductCode                0.000041
Contract_Number                                0.000014
Contract_Description                           0.000014
Contract_Supplier_Code                         0.000014
Contract_Supplier_Name                         0.000014
Customer_Name                                  0

Contract_Expression                0.120821
Contract_D_ContractNumber          0.057011
Product_ManufacturerProductCode    0.000057
Contract_Description               0.000019
Contract_Number                    0.000019
Contract_Supplier_Name             0.000019
Contract_Supplier_Code             0.000019
Product_Category                   0.000000
Date                               0.000000
Contract_ValidTo                   0.000000
Contract_ValidFrom                 0.000000
UnitClaimAmount                    0.000000
TotalClaimAmount                   0.000000
dtype: float64

Viewing date range of data


In [7]:
print("Support data date range:")
display(cust["Date"].min(), cust["Date"].max())

print("Derived supplier support view date range:")
display(supp["Date"].min(), supp["Date"].max())


Support data date range:


Timestamp('2024-02-26 00:00:00')

Timestamp('2026-06-18 00:00:00')

Derived supplier support view date range:


Timestamp('2024-02-26 00:00:00')

Timestamp('2026-06-18 00:00:00')

## Transaction Types

The full support file is observed first so the source coverage is visible. The project analysis window is then isolated to 1 January 2026 through 31 May 2026, matching the period used by the later 2026 claimback analysis. The table below focuses on the main sales transaction types: `SL/Del`, `SL/Inv`, and `SL/Crn`.


In [8]:
analysis_start = pd.Timestamp("2026-01-01")
analysis_end = pd.Timestamp("2026-05-31")
transaction_types = ["SL/Del", "SL/Inv", "SL/Crn"]

support_observation = cust.copy()
support_observation["Date"] = pd.to_datetime(support_observation["Date"], dayfirst=True, errors="coerce")
support_observation["TotalClaimAmount"] = pd.to_numeric(support_observation["TotalClaimAmount"], errors="coerce").fillna(0)

support_2026_to_may = support_observation[
    support_observation["Date"].between(analysis_start, analysis_end, inclusive="both")
].copy()

period_overview = pd.DataFrame([
    {
        "View": "Full source data",
        "Rows": len(support_observation),
        "First Date": support_observation["Date"].min(),
        "Last Date": support_observation["Date"].max(),
        "TotalClaimAmount": support_observation["TotalClaimAmount"].sum(),
    },
    {
        "View": "2026 Jan to end May",
        "Rows": len(support_2026_to_may),
        "First Date": support_2026_to_may["Date"].min(),
        "Last Date": support_2026_to_may["Date"].max(),
        "TotalClaimAmount": support_2026_to_may["TotalClaimAmount"].sum(),
    },
])

transaction_type_amounts = (
    support_2026_to_may[support_2026_to_may["SourceTransactionType"].isin(transaction_types)]
    .groupby("SourceTransactionType", dropna=False)
    .agg(
        Rows=("SourceTransactionType", "size"),
        TotalClaimAmount=("TotalClaimAmount", "sum"),
        FirstDate=("Date", "min"),
        LastDate=("Date", "max"),
    )
    .reindex(transaction_types)
    .fillna({"Rows": 0, "TotalClaimAmount": 0})
    .reset_index()
)
transaction_type_amounts["Rows"] = transaction_type_amounts["Rows"].astype(int)
transaction_type_amounts["TotalClaimAmount"] = transaction_type_amounts["TotalClaimAmount"].round(2)

display(period_overview)
display(transaction_type_amounts)


,View,Rows,First Date,Last Date,TotalClaimAmount
0,Full source data,73320,2024-02-26,2026-06-18,2.507705e+06
1,2026 Jan to end May,15842,2026-01-05,2026-05-30,6.130629e+05


,SourceTransactionType,Rows,TotalClaimAmount,FirstDate,LastDate
0,SL/Del,11864,537717.22,2026-01-05,2026-05-30
1,SL/Inv,3939,76539.85,2026-01-05,2026-05-30
2,SL/Crn,39,-1194.17,2026-01-08,2026-05-25


### Transaction Type Result

For the 2026 analysis window from 1 January 2026 to 31 May 2026, the isolated sales transaction type amounts are:

| SourceTransactionType | Rows | TotalClaimAmount |
|---|---:|---:|
| SL/Del | 11,864 | 537,717.22 |
| SL/Inv | 3,939 | 76,539.85 |
| SL/Crn | 39 | -1,194.17 |

The full source file contains 73,320 rows from 26 February 2024 to 18 June 2026 with total claim amount 2,507,705.39. The Jan-to-end-May 2026 slice contains 15,842 rows from 5 January 2026 to 30 May 2026 with total claim amount 613,062.91.


The support data spans the IQ trading/claim period available in `SupportData.xlsx`


In [9]:
display(cust["Status"].value_counts())

Status
Claimed      73317
Unclaimed        3
Name: count, dtype: int64

Claim status is abundant for the "Claimed" values


In [10]:
cust["TotalClaimAmount"].describe()

count    73320.000000
mean        34.202201
std        169.969396
min      -1741.740000
25%          1.448700
50%          3.642000
75%         11.628000
max      18500.000000
Name: TotalClaimAmount, dtype: float64

In [11]:
print("Derived views are kept in memory for notebook display")
print("Customer claim view:", cust.shape)
print("Supplier support view:", supp.shape)
display(cust.head())
display(supp.head())
display(support_clean.head())


Derived views are kept in memory for notebook display
Customer claim view: (73320, 41)
Supplier support view: (53095, 13)


,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,...,SalesDeliveryNote_NetAmountLessDiscountBase,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name
0,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS1228,38300,86.0,SL/Del,...,392.18,6.235348,14.21,4.614158,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
1,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS12S28,38322,86.0,SL/Del,...,392.18,8.130964,18.53,6.016913,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
2,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS62822,38204,86.0,SL/Del,...,392.18,5.432344,12.38,4.019935,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
3,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252815,38492,86.0,SL/Del,...,392.18,11.351756,25.87,8.400299,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
4,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252215,38490,86.0,SL/Del,...,392.18,4.058900,9.25,3.003586,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1


,Date,Contract_Number,Contract_D_ContractNumber,Product_ManufacturerProductCode,Product_Category,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name,UnitClaimAmount,TotalClaimAmount
0,2024-02-26,86.0,039-C-0692,38300,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1.366861,8.2012
1,2024-02-26,86.0,039-C-0692,38322,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1.783139,3.5663
2,2024-02-26,86.0,039-C-0692,38204,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,1.190942,2.3819
3,2024-02-26,86.0,039-C-0692,38492,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,2.488979,9.9559
4,2024-02-26,86.0,039-C-0692,38490,PRSCU,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1,0.890474,5.3428


,Date,Month,Year,SalesDeliveryNote_Branch,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Contract_Number,SourceTransactionType,...,SalesDeliveryNote_NetAmountLessDiscountBase,Product_CSQL_InvoiceCostForbranch,Product_CSQL_ListPriceForBranch,Product_CALC_FixedCostForBranch,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,Contract_Supplier_Code,Contract_Supplier_Name
0,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS1228,38300,86.0,SL/Del,...,392.18,6.235348,14.21,4.614158,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
1,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS12S28,38322,86.0,SL/Del,...,392.18,8.130964,18.53,6.016913,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
2,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS62822,38204,86.0,SL/Del,...,392.18,5.432344,12.38,4.019935,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
3,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252815,38492,86.0,SL/Del,...,392.18,11.351756,25.87,8.400299,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
4,2024-02-26,NaN,NaN,2.0,M3M500,M3 Mechanical Limited,YXS252215,38490,86.0,SL/Del,...,392.18,4.058900,9.25,3.003586,M3 MECHANICAL - YX,[D_InvoiceCost]*0.2490,01/01/0001,01/01/0001,U106,Supplier1
